In [ ]:
# Parameters: Fabric pipelines override these through the
# notebook activity's base parameters. Never put secrets here.

mode = "incremental"     # "incremental" or "recover"
recover_run_id = ""      # Bronze run ID to reprocess in recover mode

owner = "microsoft"
repository_name = "vscode"

# Environment-specific. Blank values are read from the attached
# Fabric environment's Spark properties (see docs/fabric-setup.md).
key_vault_url = ""
github_token_secret_name = ""

api_base_url = "https://api.github.com"
api_version = "2026-03-10"
api_timeout_seconds = 30
per_page = 100
max_pages = 100
max_attempts = 4
initial_backoff_seconds = 1
max_rate_limit_wait_seconds = 120

initial_lookback_days = 7
watermark_overlap_minutes = 5


In [ ]:
import json
import logging
import uuid

from datetime import datetime, timedelta, timezone
from pathlib import Path

from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.types import (
    LongType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

from builtin.api_ingestion.client import GitHubAPIClient
from builtin.api_ingestion.fabric_policy import (
    candidate_watermark,
    classify_bronze_runs,
    merge_update_condition,
    select_latest_versions,
)
from builtin.api_ingestion.transform import (
    is_pull_request,
    transform_issue,
)
from builtin.api_ingestion.validation import (
    RecordValidationError,
    parse_utc_timestamp,
)

# Silver timestamps and the checkpoint are interpreted as UTC.
spark.conf.set("spark.sql.session.timeZone", "UTC")

logger = logging.getLogger("nb_github_issues_ingestion")

if not logger.handlers:
    handler = logging.StreamHandler()
    handler.setFormatter(logging.Formatter(
        "%(asctime)s %(levelname)s %(name)s | %(message)s"
    ))
    logger.addHandler(handler)
    logger.setLevel(logging.INFO)
    logger.propagate = False


def resolve_setting(value, spark_property):
    """Prefer an explicit parameter, else the environment property."""

    resolved = (value or "").strip() or spark.conf.get(
        spark_property, ""
    ).strip()

    if not resolved:
        raise ValueError(
            f"Missing runtime setting: pass a notebook parameter or "
            f"set Spark property {spark_property} on the environment"
        )

    return resolved


LAKEHOUSE_FILES = Path("/lakehouse/default/Files")

config = {
    "owner": owner,
    "repository": repository_name,
    "key_vault_url": key_vault_url,
    "github_token_secret_name": github_token_secret_name,
    "api_base_url": api_base_url,
    "api_version": api_version,
    "api_timeout_seconds": int(api_timeout_seconds),
    "per_page": int(per_page),
    "max_pages": int(max_pages),
    "max_attempts": int(max_attempts),
    "initial_backoff_seconds": float(initial_backoff_seconds),
    "max_rate_limit_wait_seconds": float(max_rate_limit_wait_seconds),
    "initial_lookback_days": int(initial_lookback_days),
    "watermark_overlap_minutes": int(watermark_overlap_minutes),
    "bronze_dir": LAKEHOUSE_FILES / "bronze" / "github_issues",
    "bronze_archive_dir": (
        LAKEHOUSE_FILES / "bronze" / "github_issues" / "_archive"
    ),
    "quarantine_dir": LAKEHOUSE_FILES / "quarantine" / "github_issues",
    "silver_table": "silver_github_issues",
    "state_table": "ctl_ingestion_state",
    "runs_table": "ctl_ingestion_runs",
}

assert LAKEHOUSE_FILES.exists(), "Default Lakehouse Files path is unavailable"

In [ ]:
RUN_LOG_FIELDS = [
    ("run_id", "STRING"),
    ("source", "STRING"),
    ("status", "STRING"),
    ("started_at", "TIMESTAMP"),
    ("completed_at", "TIMESTAMP"),
    ("bronze_records", "BIGINT"),
    ("silver_candidates", "BIGINT"),
    ("rejected_records", "BIGINT"),
    ("error_message", "STRING"),
    # Added by the hardening pass; appended to existing tables.
    ("mode", "STRING"),
    ("failed_stage", "STRING"),
    ("bronze_path", "STRING"),
    ("recovered_from_run_id", "STRING"),
]

RUN_LOG_DDL = ", ".join(f"{name} {kind}" for name, kind in RUN_LOG_FIELDS)

SILVER_DDL = """
    issue_id BIGINT NOT NULL,
    issue_number BIGINT NOT NULL,
    repository STRING NOT NULL,
    title STRING NOT NULL,
    state STRING NOT NULL,
    author_login STRING,
    labels_json STRING NOT NULL,
    created_at TIMESTAMP NOT NULL,
    updated_at TIMESTAMP NOT NULL,
    closed_at TIMESTAMP,
    ingested_at TIMESTAMP NOT NULL,
    source_run_id STRING NOT NULL
"""

SILVER_SCHEMA = StructType([
    StructField("issue_id", LongType(), False),
    StructField("issue_number", LongType(), False),
    StructField("repository", StringType(), False),
    StructField("title", StringType(), False),
    StructField("state", StringType(), False),
    StructField("author_login", StringType(), True),
    StructField("labels_json", StringType(), False),
    StructField("created_at", TimestampType(), False),
    StructField("updated_at", TimestampType(), False),
    StructField("closed_at", TimestampType(), True),
    StructField("ingested_at", TimestampType(), False),
    StructField("source_run_id", StringType(), False),
])

CHECKPOINT_DDL = """
    source STRING,
    watermark TIMESTAMP,
    last_run_id STRING,
    last_success_at TIMESTAMP
"""


def ensure_control_tables(cfg):
    """Create tables and add run-log columns missing from older runs."""

    spark.sql(
        f"CREATE TABLE IF NOT EXISTS {cfg['silver_table']} "
        f"({SILVER_DDL}) USING DELTA"
    )
    spark.sql(
        f"CREATE TABLE IF NOT EXISTS {cfg['state_table']} "
        f"({CHECKPOINT_DDL}) USING DELTA"
    )
    spark.sql(
        f"CREATE TABLE IF NOT EXISTS {cfg['runs_table']} "
        f"({RUN_LOG_DDL}) USING DELTA"
    )

    existing = set(spark.table(cfg["runs_table"]).columns)
    missing = [
        f"{name} {kind}"
        for name, kind in RUN_LOG_FIELDS
        if name not in existing
    ]

    if missing:
        spark.sql(
            f"ALTER TABLE {cfg['runs_table']} "
            f"ADD COLUMNS ({', '.join(missing)})"
        )


def write_run(cfg, run):
    """Insert or update one run row, keyed by run_id."""

    row = spark.createDataFrame(
        [tuple(run.get(name) for name, _ in RUN_LOG_FIELDS)],
        schema=RUN_LOG_DDL,
    )

    (
        DeltaTable.forName(spark, cfg["runs_table"]).alias("t")
        .merge(row.alias("s"), "t.run_id = s.run_id")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )


def mark_interrupted_runs(cfg, source):
    """Close runs left RUNNING by a killed session.

    Assumes one execution per source at a time.
    """

    DeltaTable.forName(spark, cfg["runs_table"]).update(
        condition=(
            (F.col("source") == source)
            & (F.col("status") == "RUNNING")
        ),
        set={
            "status": F.lit("INTERRUPTED"),
            "completed_at": F.current_timestamp(),
            "error_message": F.coalesce(
                F.col("error_message"),
                F.lit("Execution ended without recording a final status"),
            ),
        },
    )


def load_run_rows(cfg, source):
    rows = (
        spark.table(cfg["runs_table"])
        .where(F.col("source") == source)
        .select("run_id", "status", "started_at", "recovered_from_run_id")
        .collect()
    )

    return [
        {
            **row.asDict(),
            "started_at": (
                row["started_at"].replace(tzinfo=timezone.utc)
                if row["started_at"] is not None
                else datetime.min.replace(tzinfo=timezone.utc)
            ),
        }
        for row in rows
    ]


def read_watermark(cfg, source):
    rows = (
        spark.table(cfg["state_table"])
        .where(F.col("source") == source)
        .select("watermark")
        .limit(1)
        .collect()
    )

    if rows and rows[0]["watermark"] is not None:
        return rows[0]["watermark"].replace(tzinfo=timezone.utc)

    return None

In [ ]:
def get_github_token(cfg):
    token = notebookutils.credentials.getSecret(
        resolve_setting(
            cfg["key_vault_url"],
            "spark.api_ingestion.key_vault_url",
        ),
        resolve_setting(
            cfg["github_token_secret_name"],
            "spark.api_ingestion.github_token_secret_name",
        ),
    )

    if not token:
        raise RuntimeError("GitHub API token not returned by Key Vault")

    return token


def bronze_file(cfg, bronze_run_id):
    # Validating the UUID also prevents path traversal in recover mode.
    return cfg["bronze_dir"] / (
        f"github_issues_{uuid.UUID(bronze_run_id)}.json"
    )


def load_bronze(path):
    with path.open(encoding="utf-8") as file:
        records = json.load(file)

    if not isinstance(records, list):
        raise ValueError(f"Bronze file is not a JSON array: {path}")

    return records


def extract_to_bronze(cfg, source, token, run_id, run_started):
    """Extract changed issues and land them as one Bronze file."""

    previous = read_watermark(cfg, source)

    if previous is not None:
        since_time = previous - timedelta(
            minutes=cfg["watermark_overlap_minutes"]
        )
    else:
        since_time = run_started - timedelta(
            days=cfg["initial_lookback_days"]
        )

    since = since_time.strftime("%Y-%m-%dT%H:%M:%SZ")

    client = GitHubAPIClient(
        base_url=cfg["api_base_url"],
        token=token,
        timeout=cfg["api_timeout_seconds"],
        api_version=cfg["api_version"],
        max_attempts=cfg["max_attempts"],
        initial_backoff=cfg["initial_backoff_seconds"],
        max_rate_limit_wait=cfg["max_rate_limit_wait_seconds"],
    )

    try:
        login = client.verify_authentication()["login"]
        logger.info("Authenticated | login=%s | since=%s", login, since)

        records = []
        pages = 0

        # get_issues raises when max_pages is exceeded, so a Bronze
        # file always holds a complete extraction window.
        for page in client.get_issues(
            owner=cfg["owner"],
            repository=cfg["repository"],
            per_page=cfg["per_page"],
            state="all",
            since=since,
            max_pages=cfg["max_pages"],
        ):
            pages += 1

            for record in page:
                try:
                    updated = parse_utc_timestamp(
                        record.get("updated_at")
                        if isinstance(record, dict) else None,
                        "updated_at",
                    )
                except RecordValidationError:
                    # Keep it so validation quarantines it visibly.
                    records.append(record)
                    continue

                # Changes after run start belong to the next window.
                if updated <= run_started:
                    records.append(record)

        requests_made = client.request_count
        retries = client.retry_count
    finally:
        client.close()

    path = bronze_file(cfg, run_id)
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("w", encoding="utf-8") as file:
        json.dump(records, file, ensure_ascii=False)

    # Read back so the run log only records a complete file.
    if len(load_bronze(path)) != len(records):
        raise RuntimeError(f"Bronze readback mismatch: {path}")

    logger.info(
        "Bronze written | path=%s | pages=%d | records=%d "
        "| requests=%d | retries=%d",
        path, pages, len(records), requests_made, retries,
    )

    return path, records


def resolve_recovery_target(cfg, source, bronze_run_id):
    """Locate an unrecovered Bronze file and its original timing."""

    path = bronze_file(cfg, bronze_run_id)

    if not path.is_file():
        raise FileNotFoundError(f"Bronze file not found: {path}")

    run_rows = load_run_rows(cfg, source)

    if bronze_run_id not in classify_bronze_runs([bronze_run_id], run_rows):
        raise RuntimeError(
            f"Bronze run {bronze_run_id} is already committed; "
            "nothing to recover"
        )

    original = [r for r in run_rows if r["run_id"] == bronze_run_id]

    # Original observation time keeps the same-timestamp policy
    # from letting an old file overwrite a newer observation.
    if original:
        ingested_at = min(r["started_at"] for r in original)
    else:
        ingested_at = datetime.fromtimestamp(
            path.stat().st_mtime, tz=timezone.utc
        )

    return path, ingested_at


def transform_records(records, source, ingested_at, bronze_run_id):
    valid = []
    rejected = []
    pull_requests = 0

    for index, record in enumerate(records):
        if is_pull_request(record):
            pull_requests += 1
            continue

        try:
            valid.append(transform_issue(
                record=record,
                repository=source,
                ingested_at=ingested_at,
                run_id=bronze_run_id,
            ))
        except (RecordValidationError, KeyError, TypeError) as exc:
            rejected.append({
                "bronze_run_id": bronze_run_id,
                "record_index": index,
                "reason": str(exc),
                "record": record,
            })

    logger.info(
        "Transformed | source_records=%d | valid=%d "
        "| pull_requests_skipped=%d | rejected=%d",
        len(records), len(valid), pull_requests, len(rejected),
    )

    return valid, rejected


def write_quarantine(cfg, run_id, rejected):
    if not rejected:
        return None

    path = cfg["quarantine_dir"] / f"rejected_{run_id}.jsonl"
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("w", encoding="utf-8") as file:
        for record in rejected:
            file.write(json.dumps(record, default=str) + "\n")

    logger.warning("Quarantine written | path=%s", path)

    return path


def merge_silver(cfg, candidates):
    """Upsert candidates using the documented same-timestamp policy."""

    incoming = spark.createDataFrame(candidates, schema=SILVER_SCHEMA)

    (
        DeltaTable.forName(spark, cfg["silver_table"]).alias("t")
        .merge(incoming.alias("s"), "t.issue_id = s.issue_id")
        .whenMatchedUpdateAll(condition=merge_update_condition("s", "t"))
        .whenNotMatchedInsertAll()
        .execute()
    )

    return incoming


def verify_silver(cfg, incoming):
    """Every candidate must be present and not newer than Silver."""

    target = spark.table(cfg["silver_table"])

    pending = (
        incoming.alias("s")
        .join(
            target.alias("t"),
            F.col("s.issue_id") == F.col("t.issue_id"),
            "left",
        )
        .where(F.expr(
            f"t.issue_id IS NULL OR ({merge_update_condition('s', 't')})"
        ))
        .count()
    )

    if pending:
        raise RuntimeError(
            f"{pending} Silver candidates were not committed"
        )

    duplicates = (
        target.groupBy("issue_id").count()
        .where(F.col("count") > 1)
        .limit(1)
        .count()
    )

    if duplicates:
        raise RuntimeError("Silver contains duplicate issue_id values")


def advance_checkpoint(cfg, source, watermark, run_id):
    """Move the watermark forward only; never backward."""

    checkpoint = spark.createDataFrame(
        [(source, watermark, run_id, datetime.now(timezone.utc))],
        schema=CHECKPOINT_DDL,
    )

    (
        DeltaTable.forName(spark, cfg["state_table"]).alias("t")
        .merge(checkpoint.alias("s"), "t.source = s.source")
        .whenMatchedUpdate(
            condition="s.watermark > t.watermark OR t.watermark IS NULL",
            set={
                "watermark": "s.watermark",
                "last_run_id": "s.last_run_id",
                "last_success_at": "s.last_success_at",
            },
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

    stored = read_watermark(cfg, source)

    if stored is None or stored < watermark:
        raise RuntimeError(
            f"Checkpoint verification failed: stored={stored}, "
            f"expected>={watermark}"
        )

    logger.info("Checkpoint | watermark=%s", stored)

In [ ]:
def run_ingestion(cfg, mode="incremental", recover_run_id=None):
    """Execute one tracked Bronze → Silver run.

    The checkpoint advances only after every Silver candidate is
    verified as committed and no records were quarantined. Failures
    are recorded in the run log with the failing stage, then
    re-raised so the notebook (and any pipeline) fails.
    """

    if mode not in {"incremental", "recover"}:
        raise ValueError(f"Invalid mode: {mode}")

    if (mode == "recover") != bool(recover_run_id):
        raise ValueError(
            "recover_run_id is required in recover mode "
            "and not allowed otherwise"
        )

    source = f"{cfg['owner']}/{cfg['repository']}"
    run_id = str(uuid.uuid4())

    run = {
        "run_id": run_id,
        "source": source,
        "status": "RUNNING",
        "started_at": datetime.now(timezone.utc),
        "mode": mode,
        "recovered_from_run_id": recover_run_id,
    }

    stage = "setup"

    logger.info(
        "Run started | run_id=%s | mode=%s | source=%s",
        run_id, mode, source,
    )

    try:
        ensure_control_tables(cfg)
        mark_interrupted_runs(cfg, source)
        write_run(cfg, run)

        if mode == "incremental":
            stage = "bronze"
            token = get_github_token(cfg)
            bronze_path, records = extract_to_bronze(
                cfg, source, token, run_id, run["started_at"]
            )
            bronze_run_id = run_id
            ingested_at = run["started_at"]
        else:
            stage = "recovery_load"
            bronze_path, ingested_at = resolve_recovery_target(
                cfg, source, recover_run_id
            )
            records = load_bronze(bronze_path)
            bronze_run_id = recover_run_id

        run["bronze_path"] = str(bronze_path)
        run["bronze_records"] = len(records)
        write_run(cfg, run)

        stage = "transform"
        valid, rejected = transform_records(
            records, source, ingested_at, bronze_run_id
        )
        candidates = select_latest_versions(valid)
        run["silver_candidates"] = len(candidates)
        run["rejected_records"] = len(rejected)
        quarantine_path = write_quarantine(cfg, run_id, rejected)

        if candidates:
            stage = "silver_merge"
            incoming = merge_silver(cfg, candidates)

            stage = "silver_verify"
            verify_silver(cfg, incoming)

        stage = "checkpoint"

        # Valid records stay merged (MERGE is idempotent), but the
        # window is re-extracted until rejected records are resolved.
        if rejected:
            raise RuntimeError(
                f"{len(rejected)} rejected records in {quarantine_path} "
                "require review before the checkpoint can advance"
            )

        watermark = candidate_watermark(candidates)

        if watermark is not None:
            advance_checkpoint(cfg, source, watermark, run_id)

        run["status"] = "SUCCESS"
        stage = None

        logger.info(
            "Run succeeded | run_id=%s | candidates=%d | watermark=%s",
            run_id, len(candidates), watermark,
        )

        return {
            "run_id": run_id,
            "status": "SUCCESS",
            "mode": mode,
            "bronze_path": str(bronze_path),
            "bronze_records": len(records),
            "silver_candidates": len(candidates),
            "watermark": watermark,
        }

    except Exception as exc:
        run["status"] = "FAILED"
        run["failed_stage"] = stage
        run["error_message"] = f"{type(exc).__name__}: {exc}"[:4000]

        logger.exception(
            "Run failed | run_id=%s | stage=%s", run_id, stage
        )
        raise

    finally:
        # Cancellation (KeyboardInterrupt) bypasses the handler above.
        if run["status"] == "RUNNING":
            run["status"] = "INTERRUPTED"
            run["failed_stage"] = stage

        run["completed_at"] = datetime.now(timezone.utc)

        try:
            write_run(cfg, run)
        except Exception:
            logger.exception(
                "Could not record final run status | run_id=%s", run_id
            )

            # Never mask the original failure, but do not report
            # success when the outcome could not be recorded.
            if run["status"] == "SUCCESS":
                raise

In [ ]:
def unrecovered_bronze_files(cfg):
    """Bronze files whose data no successful run has committed."""

    source = f"{cfg['owner']}/{cfg['repository']}"

    files = {
        path.stem.removeprefix("github_issues_"): path
        for path in cfg["bronze_dir"].glob("github_issues_*.json")
    }

    pending = classify_bronze_runs(
        sorted(files), load_run_rows(cfg, source)
    )

    return [
        {
            "bronze_run_id": run_id,
            "run_status": status,
            "size_bytes": files[run_id].stat().st_size,
            "modified_at": datetime.fromtimestamp(
                files[run_id].stat().st_mtime, tz=timezone.utc
            ),
        }
        for run_id, status in sorted(pending.items())
    ]


def archive_bronze_file(cfg, bronze_run_id, reason):
    """Move an unrecovered Bronze file out of the landing folder.

    Use for files that are corrupt or deliberately not reprocessed.
    The move is recorded in the run log as an ARCHIVED row.
    """

    source = f"{cfg['owner']}/{cfg['repository']}"
    path = bronze_file(cfg, bronze_run_id)

    if not path.is_file():
        raise FileNotFoundError(f"Bronze file not found: {path}")

    if bronze_run_id not in classify_bronze_runs(
        [bronze_run_id], load_run_rows(cfg, source)
    ):
        raise RuntimeError(
            f"Bronze run {bronze_run_id} is committed; refusing to archive"
        )

    destination = cfg["bronze_archive_dir"] / path.name
    destination.parent.mkdir(parents=True, exist_ok=True)

    if destination.exists():
        raise FileExistsError(f"Archive already contains {destination}")

    path.rename(destination)

    now = datetime.now(timezone.utc)

    write_run(cfg, {
        "run_id": str(uuid.uuid4()),
        "source": source,
        "status": "ARCHIVED",
        "started_at": now,
        "completed_at": now,
        "mode": "archive",
        "bronze_path": str(destination),
        "recovered_from_run_id": bronze_run_id,
        "error_message": reason,
    })

    logger.warning(
        "Bronze archived | run_id=%s | path=%s | reason=%s",
        bronze_run_id, destination, reason,
    )

    return destination

In [ ]:
result = run_ingestion(
    config,
    mode=mode,
    recover_run_id=recover_run_id.strip() or None,
)

result_json = json.dumps(result, default=str)

print("Execution result:")
print(result_json)

In [ ]:
pending_bronze = unrecovered_bronze_files(config)

if pending_bronze:
    logger.warning(
        "%d Bronze file(s) await recovery: %s",
        len(pending_bronze),
        ", ".join(f["bronze_run_id"] for f in pending_bronze),
    )
    display(spark.createDataFrame(pending_bronze))
else:
    print("No unrecovered Bronze files")

display(
    spark.table(config["runs_table"])
    .orderBy(F.col("started_at").desc())
    .limit(20)
)

In [ ]:
notebookutils.notebook.exit(result_json)